# Simulating Coh-Metrix-Style LSA Indices in Python

This notebook shows how to build a **Coh-Metrix-like LSA tool** in Python. It simulates the logic of Coh-Metrix LSA coherence indices by building a reference semantic space and then measuring semantic overlap among sentences and paragraphs in a target text.

Coh-Metrix uses LSA to estimate semantic relatedness among discourse units such as sentences and paragraphs. Higher LSA similarity usually suggests stronger semantic overlap and therefore stronger potential coherence.


## 1. Coh-Metrix-Like LSA Workflow

A Coh-Metrix-like LSA workflow separates the **reference semantic space** from the **target text being assessed**.

```text
reference corpus -> TF-IDF -> SVD -> fixed LSA semantic space
target text -> transform into that space -> LSA coherence indices
```

This matters because LSA needs background evidence about how words and discourse units are semantically related. If the model learns only from the short target text, the semantic space is unstable and highly dependent on that one text. A reference corpus gives the model a broader semantic map before the target text is evaluated.

In this notebook, the reference corpus is small because it is designed for classroom demonstration. For formal research, it should be replaced with a larger corpus that matches the register, genre, discipline, and language of the texts being studied.

First, a reference corpus is used to build an LSA semantic space. Then, the sentences and paragraphs of the target text are projected into that space. By calculating the cosine similarity between them, we can assess the local and global coherence of the text.


## 2. Coh-Metrix-Like LSA Indices in This Notebook

This notebook computes the following indicators:

- **Adjacent sentence LSA mean**: average similarity between neighboring sentences.
- **Adjacent sentence LSA SD**: variation in neighboring sentence similarity.
- **All sentence pairs LSA mean**: average similarity among all sentence pairs.
- **All sentence pairs LSA SD**: variation among all sentence-pair similarities.
- **Adjacent paragraph LSA mean**: average similarity between neighboring paragraphs.
- **Adjacent paragraph LSA SD**: variation in neighboring paragraph similarity.
- **Given/new LSA mean**: similarity between each sentence and the previous discourse context.
- **Given/new LSA SD**: variation in given/new similarity.

These labels are intentionally descriptive. They are **Coh-Metrix-like**, not official Coh-Metrix output labels.


In [34]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity


## 3. Target Text

The target text below contains three topical zones: machine learning, pets, and finance. The paragraph breaks are included so that we can also calculate paragraph-level LSA similarity.


In [35]:
target_text = """
Machine learning is useful for text analysis.
Text mining and machine learning are used to analyze documents.
Deep learning models can process natural language.

Dogs and cats are common household pets.
My cat likes to sleep on the sofa.

The stock market increased after the economic report.
Investors are watching the financial market closely.
"""


## 4. Reference Corpus

The reference corpus is the background material used to build the LSA semantic space. It gives the model examples of how terms tend to occur together across different semantic areas.

Adding a reference corpus is important for three reasons. First, it makes the semantic space less dependent on one short target text. Second, it makes scores more comparable across different target texts. Third, it allows the analysis to reflect a chosen discourse domain, such as academic writing, news, translation assignments, or a specialized field.

For serious research, replace this small teaching corpus with a much larger corpus that matches your teaching or research context, such as translated texts, academic writing, news reports, or domain-specific documents.

A good reference corpus should be large enough, domain-relevant, language-matched, high-quality, and processed in the same way as the target text.


In [36]:
reference_corpus = [
    # Text analysis and machine learning
    "Machine learning methods are useful for text analysis and document classification.",
    "Text mining can discover patterns in large collections of documents.",
    "Natural language processing models analyze words, sentences, and discourse.",
    "Deep learning systems can represent semantic information in language data.",
    "Researchers use computational models to compare texts and translations.",
    "Vector space models represent documents with numerical features.",
    "Semantic similarity can be measured by comparing sentence vectors.",
    "Translation quality depends partly on lexical cohesion and discourse coherence.",

    # Pets and household life
    "Dogs and cats are common household pets.",
    "Many families keep cats, dogs, and other animals at home.",
    "A cat may sleep on a sofa during the afternoon.",
    "Pet owners often care about food, comfort, and health.",
    "Domestic animals are part of everyday household life.",

    # Finance and economy
    "The stock market changed after the economic report.",
    "Investors follow financial news and market signals closely.",
    "Economic indicators can influence stock prices and investment decisions.",
    "Financial analysts compare market trends across different sectors.",
    "Reports about inflation and employment affect investor confidence.",

    # Discourse and writing
    "A coherent paragraph develops one topic through connected sentences.",
    "Writers use transitions to guide readers from old information to new information.",
    "A sudden topic shift may reduce local coherence in a text.",
    "Global coherence depends on how well the whole text maintains its theme.",
    "Readers understand a text more easily when sentences are semantically connected.",
]


## 5. Split Text into Sentences and Paragraphs

Coh-Metrix computes indices at different discourse levels. Here we use sentences and paragraphs.


In [37]:
def split_sentences(text):
    """A simple sentence splitter for English classroom examples."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [sentence.strip() for sentence in sentences if sentence.strip()]


def split_paragraphs(text):
    """Split paragraphs by blank lines."""
    paragraphs = re.split(r'\n\s*\n', text.strip())
    return [paragraph.strip() for paragraph in paragraphs if paragraph.strip()]


sentences = split_sentences(target_text)
paragraphs = split_paragraphs(target_text)

print("Sentences:")
for i, sentence in enumerate(sentences, start=1):
    print(f"S{i}: {sentence}")

print("\nParagraph count:", len(paragraphs))
for i, paragraph in enumerate(paragraphs, start=1):
    print(f"P{i}: {paragraph.replace(chr(10), ' ')}")


Sentences:
S1: Machine learning is useful for text analysis.
S2: Text mining and machine learning are used to analyze documents.
S3: Deep learning models can process natural language.
S4: Dogs and cats are common household pets.
S5: My cat likes to sleep on the sofa.
S6: The stock market increased after the economic report.
S7: Investors are watching the financial market closely.

Paragraph count: 3
P1: Machine learning is useful for text analysis. Text mining and machine learning are used to analyze documents. Deep learning models can process natural language.
P2: Dogs and cats are common household pets. My cat likes to sleep on the sofa.
P3: The stock market increased after the economic report. Investors are watching the financial market closely.


## 6. Build a Reference LSA Semantic Space

This is the key Coh-Metrix-like step. The vectorizer and SVD model are fitted on the reference corpus. The target text will later be projected into this fixed semantic space.


In [38]:
def fit_reference_lsa_space(reference_texts, n_components=8):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        lowercase=True,
        min_df=1
    )

    reference_tfidf = vectorizer.fit_transform(reference_texts)

    max_components = min(reference_tfidf.shape) - 1
    n_components = min(n_components, max_components)

    if n_components < 2:
        raise ValueError(
            "The reference corpus is too small for meaningful LSA. "
            "Use more reference texts or reduce n_components."
        )

    svd = TruncatedSVD(n_components=n_components, random_state=42)
    svd.fit(reference_tfidf)

    return vectorizer, svd, reference_tfidf


vectorizer, svd, reference_tfidf = fit_reference_lsa_space(reference_corpus, n_components=8)

print("Reference TF-IDF matrix shape:", reference_tfidf.shape)
print("Number of LSA dimensions:", svd.n_components)
print("Explained variance ratio:", round(svd.explained_variance_ratio_.sum(), 3))


Reference TF-IDF matrix shape: (23, 118)
Number of LSA dimensions: 8
Explained variance ratio: 0.424


The notebook has built a small reference semantic space from 23 reference texts and 118 terms; it compresses that vocabulary into 8 LSA dimensions, retaining about 42.4% of the reference corpus’s semantic variation.

## 7. Transform the Target Text into the Reference Space

The target sentences and paragraphs are not used to train the LSA space. They are only transformed into the semantic space learned from the reference corpus.


In [39]:
def transform_units(units, vectorizer, svd):
    tfidf = vectorizer.transform(units)
    lsa_matrix = svd.transform(tfidf)
    return lsa_matrix


sentence_lsa = transform_units(sentences, vectorizer, svd)
paragraph_lsa = transform_units(paragraphs, vectorizer, svd)

print("Sentence LSA matrix shape:", sentence_lsa.shape)
print("Paragraph LSA matrix shape:", paragraph_lsa.shape)


Sentence LSA matrix shape: (7, 8)
Paragraph LSA matrix shape: (3, 8)


The reference corpus created an 8-dimensional semantic space, and now every sentence and paragraph in the target text has been placed into that same space. This allows us to calculate sentence-to-sentence and paragraph-to-paragraph semantic similarity.

## 8. Helper Functions for LSA Similarity

Cosine similarity compares vectors in the LSA semantic space. The raw score may theoretically range from `-1` to `1`. For classroom interpretation, this notebook also provides a clipped `0-1` score, where negative values are treated as `0`.


In [40]:
def cosine_score(vector_a, vector_b):
    return cosine_similarity([vector_a], [vector_b])[0, 0]


def clipped_score(score):
    return float(np.clip(score, 0, 1))


def adjacent_scores(matrix):
    rows = []
    for i in range(len(matrix) - 1):
        raw = cosine_score(matrix[i], matrix[i + 1])
        rows.append({
            "pair": f"{i + 1}-{i + 2}",
            "raw_lsa_similarity": raw,
            "coherence_score_0_to_1": clipped_score(raw),
        })
    return pd.DataFrame(rows)


def all_pair_scores(matrix):
    rows = []
    for i in range(len(matrix)):
        for j in range(i + 1, len(matrix)):
            raw = cosine_score(matrix[i], matrix[j])
            rows.append({
                "pair": f"{i + 1}-{j + 1}",
                "raw_lsa_similarity": raw,
                "coherence_score_0_to_1": clipped_score(raw),
            })
    return pd.DataFrame(rows)


## 9. Sentence-Level LSA Indices

These indices approximate Coh-Metrix-style sentence overlap measures.

- Adjacent sentence similarity is a local coherence indicator.
- All sentence-pair similarity is a broader global coherence indicator.


In [41]:
adjacent_sentence_df = adjacent_scores(sentence_lsa)
all_sentence_pair_df = all_pair_scores(sentence_lsa)

print("Adjacent sentence LSA similarities:")
print(adjacent_sentence_df.round(3).to_string(index=False))

print("\nAll sentence-pair LSA similarities:")
print(all_sentence_pair_df.round(3).to_string(index=False))


Adjacent sentence LSA similarities:
pair  raw_lsa_similarity  coherence_score_0_to_1
 1-2               0.812                   0.812
 2-3               0.409                   0.409
 3-4               0.005                   0.005
 4-5              -0.092                   0.000
 5-6               0.032                   0.032
 6-7               0.447                   0.447

All sentence-pair LSA similarities:
pair  raw_lsa_similarity  coherence_score_0_to_1
 1-2               0.812                   0.812
 1-3               0.210                   0.210
 1-4               0.011                   0.011
 1-5              -0.480                   0.000
 1-6               0.001                   0.001
 1-7               0.092                   0.092
 2-3               0.409                   0.409
 2-4              -0.024                   0.000
 2-5               0.104                   0.104
 2-6              -0.009                   0.000
 2-7              -0.020                   0.

## 10. Paragraph-Level LSA Indices

If the text has two or more paragraphs, we can compare adjacent paragraphs. This approximates paragraph-level semantic overlap.


In [42]:
if len(paragraphs) >= 2:
    adjacent_paragraph_df = adjacent_scores(paragraph_lsa)
    print("Adjacent paragraph LSA similarities:")
    print(adjacent_paragraph_df.round(3).to_string(index=False))
else:
    adjacent_paragraph_df = pd.DataFrame(
        columns=["pair", "raw_lsa_similarity", "coherence_score_0_to_1"]
    )
    print("The text has fewer than two paragraphs, so paragraph-level LSA is not available.")


Adjacent paragraph LSA similarities:
pair  raw_lsa_similarity  coherence_score_0_to_1
 1-2              -0.012                     0.0
 2-3              -0.004                     0.0


## 11. Given/New LSA Index

The given/new idea compares each sentence with the discourse context that came before it.

In this simplified version, the previous context is represented by the average LSA vector of all earlier sentences. A high score means the current sentence is strongly connected to prior information. A low score suggests a possible topic shift or weak transition.


In [43]:
def given_new_scores(sentence_matrix):
    rows = []
    for i in range(1, len(sentence_matrix)):
        previous_context = sentence_matrix[:i].mean(axis=0)
        current_sentence = sentence_matrix[i]
        raw = cosine_score(previous_context, current_sentence)
        rows.append({
            "sentence": f"S{i + 1}",
            "compared_with": f"S1-S{i}",
            "raw_lsa_similarity": raw,
            "coherence_score_0_to_1": clipped_score(raw),
        })
    return pd.DataFrame(rows)


given_new_df = given_new_scores(sentence_lsa)

print("Given/new LSA similarities:")
print(given_new_df.round(3).to_string(index=False))


Given/new LSA similarities:
sentence compared_with  raw_lsa_similarity  coherence_score_0_to_1
      S2         S1-S1               0.812                   0.812
      S3         S1-S2               0.321                   0.321
      S4         S1-S3              -0.003                   0.000
      S5         S1-S4              -0.167                   0.000
      S6         S1-S5              -0.005                   0.000
      S7         S1-S6               0.199                   0.199


## 12. Coh-Metrix-Like LSA Summary Table

This table collects the main simulated indices. The mean shows the general level of semantic overlap. The standard deviation shows how uneven the overlap is across the text.


In [44]:
def mean_or_nan(df, column="coherence_score_0_to_1"):
    return np.nan if df.empty else df[column].mean()


def sd_or_nan(df, column="coherence_score_0_to_1"):
    return np.nan if df.empty else df[column].std(ddof=0)


summary = pd.Series({
    "LSA_adjacent_sentence_mean": mean_or_nan(adjacent_sentence_df),
    "LSA_adjacent_sentence_sd": sd_or_nan(adjacent_sentence_df),
    "LSA_all_sentence_pairs_mean": mean_or_nan(all_sentence_pair_df),
    "LSA_all_sentence_pairs_sd": sd_or_nan(all_sentence_pair_df),
    "LSA_adjacent_paragraph_mean": mean_or_nan(adjacent_paragraph_df),
    "LSA_adjacent_paragraph_sd": sd_or_nan(adjacent_paragraph_df),
    "LSA_given_new_mean": mean_or_nan(given_new_df),
    "LSA_given_new_sd": sd_or_nan(given_new_df),
})

print("Coh-Metrix-like LSA summary:")
print(summary.round(3))


Coh-Metrix-like LSA summary:
LSA_adjacent_sentence_mean     0.284
LSA_adjacent_sentence_sd       0.301
LSA_all_sentence_pairs_mean    0.105
LSA_all_sentence_pairs_sd      0.203
LSA_adjacent_paragraph_mean    0.000
LSA_adjacent_paragraph_sd      0.000
LSA_given_new_mean             0.222
LSA_given_new_sd               0.290
dtype: float64


## 13. Interpretation Guide for Translation Students

Use the scores diagnostically rather than mechanically.

A high **adjacent sentence mean** suggests smooth local semantic flow. A low score points to possible weak sentence-to-sentence connection.

A high **all sentence-pair mean** suggests stronger global topic unity. A low score may indicate multiple unrelated topics.

A high **adjacent paragraph mean** suggests that neighboring paragraphs are semantically connected. A low score may indicate abrupt paragraph-level shifts.

A high **given/new mean** suggests that new sentences build on previous discourse. A low score suggests that the sentence may introduce new information without enough connection to what came before.

For translation assessment, low-scoring transitions can be inspected for missing connectives, inconsistent terminology, unclear reference, or poor thematic progression.


## 14. Limitations

This notebook simulates Coh-Metrix-style LSA indices, but it is not the official Coh-Metrix tool.

Important limitations:

- The reference corpus here is very small.
- Coh-Metrix uses its own preprocessing pipeline and semantic spaces.
- This notebook uses a simplified sentence splitter.
- The given/new index is an approximation.
- Scores should be interpreted with human linguistic judgment.

For classroom use, the notebook is useful because it makes the logic visible. For formal research, use a larger reference corpus and validate the indices carefully.
